In [18]:
# %%writefile Ind_OBV_EX.py

import numpy as np
import pandas as pd

import QUANTAXIS as QA

import Ind_Model_Base

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import Analysis_Funs as af
import Sample_Tools as smpl
import Pretreat_Tools as pretreat


%load_ext autoreload
%autoreload 2
%aimport Analysis_Funs,Sample_Tools,Pretreat_Tools

class OBV(Ind_Model_Base.Ind_Model):
    '''能量潮指标改进版'''
    def __init__(self,data, frequence=QA.FREQUENCE.DAY):
        super().__init__(data, 'OBV_EX', frequence)
    
    def on_set_params_default(self):
        return {'SHORT':5, 'LONG':15}
        
    def on_indicator_structuring(self, data):
        return self.excute_for_multicode(data, self.kernel, **self.pramas)

    
    def on_desition_structuring(self, data, ind_data):
        """
        1.短期量价穿越长期,res为1，买入信号参考。
        2.相反则res为-1，卖出信号参考。
        """
        return pd.DataFrame({'res':ind_data['CROSS_JC'] + ind_data['CROSS_SC']*-1})
        
    def kernel(self,dataframe,SHORT=5,LONG=15):
        '''多空比率净额= [（收盘价－最低价）－（最高价-收盘价）] ÷（ 最高价－最低价）×V'''
        long_short_ratio=((dataframe.close - dataframe.low) - (dataframe.high - dataframe.close)) / (dataframe.high-dataframe.low) * dataframe.volume
        
        short =QA.EMA(long_short_ratio,SHORT)
        long = QA.EMA(long_short_ratio,LONG)
        

        CROSS_JC=QA.CROSS(short, long)
        CROSS_SC=QA.CROSS(long, short)

        return pd.DataFrame({'LSR':long_short_ratio,'CROSS_JC':CROSS_JC, 'CROSS_SC':CROSS_SC})
#         return pd.DataFrame({'LSR':long_short_ratio})
    
    
    def plot(self,figsize=(1120/72,420/72)) -> dict:
        fig = plt.figure(figsize=figsize)
        groups = self.ind_df.groupby(level=1)
        for idx,item in enumerate(groups):
            inds_ = item[1].reset_index('code',drop=True)
            ax = fig.add_subplot(len(groups),1,idx+1)
             
            inds_.plot(ax=ax,legend=True)
            ax.set_title(item[0],color='r', loc ='left', pad=-10)
            ax.xaxis.set_major_locator(ticker.MaxNLocator(10))
            plt.xticks(rotation = 0)
            
    
    def plot_mix(self,figsize=(1120/72,420/72)) -> dict:
        fig = plt.figure(figsize=figsize)
        groups = self.ind_df.groupby(level=1)
        def x1(item):
            inds_ = item.reset_index('code',drop=True)
            plt.plot(inds_['LSR'])
#             print(item.name)
        groups.apply(x1)
        plt.legend(groups.groups.keys())
        
    def self_test(self):
        data = get_sample_by_zs(name='上证50', end='2020-06-29', gap=250, only_main=True)
        df = resample_stockdata_low(data.data,freq="M")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
data = smpl.get_sample_by_zs(name='上证50', end='2020-06-29', gap=250, only_main=True)
df = smpl.resample_stockdata_low(data.data,freq="W")
ret_forward = smpl.get_forward_return(df,'close')


obv = OBV(df)
obv.fit()
ind = obv.ind_df['LSR']
factor_standardized = pretreat.standardize(ind, multi_code=True)
rank_ic = af.get_rank_ic(factor_standardized, ret_forward)
print(rank_ic)
print(af.get_ic_desc(rank_ic))
print(af.get_ic_ir(rank_ic))
print(af.get_winning_rate(rank_ic))





date
2019-06-23       -0.11409
2019-06-30        -0.4632
2019-07-07      -0.149167
2019-07-14      0.0895005
2019-07-21      -0.515726
2019-07-28     -0.0838037
2019-08-04      0.0536257
2019-08-11       -0.19226
2019-08-18      0.0242076
2019-08-25     -0.0640469
2019-09-01      -0.476769
2019-09-08     0.00955302
2019-09-15      -0.305363
2019-09-22      -0.320234
2019-09-29      -0.410769
2019-10-06      -0.413591
2019-10-13       0.223513
2019-10-20     0.00173686
2019-10-27      -0.110074
2019-11-03      -0.203105
2019-11-10      0.0129179
2019-11-17      -0.351607
2019-11-24       0.157186
2019-12-01       0.327725
2019-12-08     -0.0838037
2019-12-15       0.250868
2019-12-22       0.233391
2019-12-29       -0.17564
2020-01-05      0.0570994
2020-01-12     -0.0804386
2020-01-19       0.305797
2020-01-26           None
2020-02-02           None
2020-02-09   -0.000868527
2020-02-16      -0.384173
2020-02-23      0.0861919
2020-03-01     -0.0277898
2020-03-08     -0.0305037
2020-03